# 03 - Model Training, Tuning, and Evaluation

## Imports

In [46]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import root_mean_squared_error
from sklearn.datasets import load_diabetes
import pandas as pd 
import seaborn as sns
from sklearn.model_selection import train_test_split

# Make sure we get outputs from a probabilitistic model each time (for reproducibility) 
np.random.seed(1515)

## How can we make Generalizable Models?

### Data Splits

In our previous example, we covered how there was a difference in performance between Period 1/3 data and Period 5 data for both the good and bad models. 

If we follow the path we took for the good model specifically, we built our model on the good data from Period 3, and tested in on Period 5 data. This was done to ensure our model performed well not just on the Period 3 data, but another dataset (Period 5). 

In general, we only have one dataset to work with. So, we have to split our existing data into the following categories:

1. **Training**: this is the data we build our original model on
2. **Validation**: this is the data we alter our model on to see what the best performance is
3. **Test**: this is the data we perform a final test on to ensure generalizability

A common way to split this data is 70/15/15, which represents the percentage of total data in the training/validation/test splits. Let's use a sample dataset to show what this could look like:

In [53]:
# read in diabetes dataset from Seaborn 
diabetes = load_diabetes()

# store as pandas DataFrame
df = pd.DataFrame(data=diabetes.data, columns=diabetes.feature_names)
df['target'] = diabetes.target
df.head(5)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


This is a premade dataset that shows diabetes progression along with age, sex, BMI, Blood Pressure, and 6 Serum measurements. 'target' is the measure of the diabetes progression. 

Let's look at some basic info about this dataset: 

In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     442 non-null    float64
 1   sex     442 non-null    float64
 2   bmi     442 non-null    float64
 3   bp      442 non-null    float64
 4   s1      442 non-null    float64
 5   s2      442 non-null    float64
 6   s3      442 non-null    float64
 7   s4      442 non-null    float64
 8   s5      442 non-null    float64
 9   s6      442 non-null    float64
 10  target  442 non-null    float64
dtypes: float64(11)
memory usage: 38.1 KB


We have 442 rows, none of which are null (or no value), so we can proceed to splitting our data:

In [78]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['target']) # X is all of our independent/predictor variables
y = df['target'] # y is our target/outcome variable 

# first split: Get training data
# second split: split remaining into validation and test data

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size = 0.3, random_state = 1515)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, random_state = 1515)

X_train

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
335,0.005383,-0.044642,-0.048241,-0.012556,0.001183,-0.006637,0.063367,-0.039493,-0.051404,-0.059067
207,0.009016,-0.044642,0.045529,0.028758,0.012191,-0.013840,0.026550,-0.039493,0.046133,0.036201
187,-0.067268,-0.044642,-0.054707,-0.026328,-0.075870,-0.082106,0.048640,-0.076395,-0.086827,-0.104630
101,0.016281,0.050680,-0.045007,0.063187,0.010815,-0.000374,0.063367,-0.039493,-0.030748,0.036201
338,-0.063635,-0.044642,-0.033151,-0.033213,0.001183,0.024051,-0.024993,-0.002592,-0.022517,-0.059067
...,...,...,...,...,...,...,...,...,...,...
351,-0.085430,0.050680,-0.040696,-0.033213,-0.081374,-0.069580,-0.006584,-0.039493,-0.057803,-0.042499
119,0.016281,-0.044642,-0.047163,-0.002228,-0.019456,-0.042963,0.033914,-0.039493,0.027364,0.027917
14,0.045341,-0.044642,-0.025607,-0.012556,0.017694,-0.000061,0.081775,-0.039493,-0.031988,-0.075636
326,0.045341,0.050680,-0.008362,-0.033213,-0.007073,0.001191,-0.039719,0.034309,0.029935,0.027917


We have split our data successfully! Now, how do we go about building our model? 

### General Model Selection